# Import Libraries

In [50]:
import pandas as pd
import numpy as np
import h5py
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, cross_val_score
import matplotlib.pyplot as plt
import os
from transformers import BertTokenizer, TFBertModel, TFBertForSequenceClassification, AutoTokenizer, TFAutoModelForSequenceClassification
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, Dense, Conv1D, GlobalMaxPooling1D, SimpleRNN, MultiHeadAttention
from tensorflow.keras.layers import LSTM, Input, BatchNormalization, Dropout, Attention, GlobalAveragePooling1D
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalAccuracy
import torch
from collections import Counter

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

In [51]:
def preprocess_text(text):
    # Define interrogative words to KEEP
    interrogatives = {"what", "why", "how", "who", "where", "when", "which", "whom", "whose", "no", "not",
                    "very" ,"too" ,"too" ,"just", "if", "but", "however", "without", "like"}
    custom_stopwords = set(nlp.Defaults.stop_words)
    custom_stopwords -= interrogatives

    doc = nlp(text.lower().strip())  # Lowercase and remove whitespace
    
# Process tokens: lemmatize, filter stopwords/punct/numbers, keep interrogatives
    tokens = [
        token.lemma_ 
        for token in doc 
        if (
            (not token.is_stop or token.text in interrogatives) and  # Keep interrogatives
            not token.is_punct and token.is_alpha                                  # Remove punctuation
            # (token.is_alpha or token.like_num)                       # Keep words/numbers
        )
    ]

    return ' '.join(tokens)

In [52]:
def make_dataset(texts, labels, tokenizer, shuffle=False, max_len= 20, batch_size = 32):
    def gen():
        for t, l in zip(texts, labels):
            enc = tokenizer(
                t,
                truncation=True,
                padding='max_length',
                max_length=max_len,
                return_tensors='tf'
            )
            yield ({'input_ids': enc['input_ids'][0], 'attention_mask': enc['attention_mask'][0]}, l)

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            {'input_ids': tf.TensorSpec(shape=(max_len,), dtype=tf.int32),
             'attention_mask': tf.TensorSpec(shape=(max_len,), dtype=tf.int32)},
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=len(texts))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Pre-Processing

### Import Data

In [ ]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

# Load dataset
df = pd.DataFrame()
for i in [2,3,4,5]:
    q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(i) + '.csv')
    q_df['dataset_id'] = i
    df = pd.concat([df , q_df], )
    
df = df.reset_index(drop=True)

# Apply preprocessing
df['label'] = df['label'].str.lower()
df['label'] = df['label'].replace(mapping)

df['processed_question'] = df['question'].apply(preprocess_text)
df['processed_question'] = [''.join(text) for text in df['processed_question']]

In [ ]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

# Load dataset
test_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(1) + '.csv')
    
test_df = test_df.reset_index(drop=True)

# Apply preprocessing
test_df['label'] = test_df['label'].str.lower()
test_df['label'] = test_df['label'].replace(mapping)

test_df['processed_question'] = test_df['question'].apply(preprocess_text)
test_df['processed_question'] = [''.join(text) for text in test_df['processed_question']]

## Tokenize
### Execute any one

### TF-IDF

In [ ]:
# token_pattern: words of 2+ word-chars, but not all digits
token_pattern = r"(?u)\b[a-zA-Z]{3,}\b"

vectorizer = TfidfVectorizer(
    min_df= 5 , ngram_range=(1,2), use_idf= True, norm= 'l2', token_pattern= token_pattern
                             )
tfidf = vectorizer.fit_transform(df['processed_question'])

# Convert TF-IDF sparse matrix to DataFrame with appropriate column names
tfidf_df = pd.DataFrame(tfidf.toarray(), columns= vectorizer.get_feature_names_out(), index=df.index)

print('Training Shape: ' + str(tfidf_df.shape))

In [ ]:
test_tfidf = vectorizer.transform(test_df['processed_question'])

# Convert TF-IDF sparse matrix to DataFrame with appropriate column names
test_tfidf_df = pd.DataFrame(test_tfidf.toarray(), columns= vectorizer.get_feature_names_out(), index=test_df.index)

print('Test Shape: ' + str(test_tfidf_df.shape))

In [ ]:
x_train , y_train = tfidf_df , df['label']
x_test , y_test = test_tfidf , test_df['label']

### Classical Tokenizer

In [55]:
num_size = 20000

tokenizer = Tokenizer(num_words=num_size, oov_token='<OOV>')
tokenizer.fit_on_texts(df['processed_question'])
sequences = tokenizer.texts_to_sequences(df['processed_question'])

In [56]:
max_len = 0

for seq in sequences:
    max_len = max(max_len , len(seq))

print(max_len)

data = pad_sequences(sequences, maxlen=max_len)

41


In [57]:
tokenizer.fit_on_texts(test_df['processed_question'])
test_sequences = tokenizer.texts_to_sequences(test_df['processed_question'])
test_data = pad_sequences(test_sequences, maxlen=max_len)

# Modelling

## Basic Deep Learning

### IF TF-IDF used

### If vector embedding used

In [58]:
y_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

y_mapped = df['label'].map(y_mapper)
y_test_mapped = test_df['label'].map(y_mapper)

In [59]:
x_train , y_train = data , to_categorical(y_mapped , 6)
x_test, y_test = test_data , to_categorical(y_test_mapped , 6)

In [11]:
x , y = data , to_categorical(y_mapped , 6)
x_train , x_test , y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=34 , stratify= y_mapped)

In [60]:
EMBEDDING_DIM = 28
word_index = tokenizer.word_index
VOCAB_SIZE = len(word_index) + 1

## 1D CNN

In [61]:
cnn_model = Sequential([
    Input(shape=(max_len,)),
    Embedding(VOCAB_SIZE, EMBEDDING_DIM),

    Conv1D(128, 5, activation='gelu', padding= 'same'),
    BatchNormalization(),
    GlobalMaxPooling1D(),
    Dropout(0.3),

    Dense(64, activation='sigmoid'),
    Dropout(0.4),

    Dense(6, activation='softmax')
])
cnn_model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=1e-4), metrics=['accuracy'])

cnn_model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 41, 28)         │       109,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 41, 128)        │        18,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 41, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_3          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 136,854 (534.59 KB)

 Trainable params: 136,598 (533.59 KB)

 Non-trainable params: 256 (1.00 KB)

In [62]:
cnn_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.8)

Epoch 1/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.1328 - loss: 2.0767 - val_accuracy: 0.1300 - val_loss: 1.7903
Epoch 2/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1853 - loss: 1.9342 - val_accuracy: 0.1300 - val_loss: 1.7696
Epoch 3/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1457 - loss: 1.9886 - val_accuracy: 0.1300 - val_loss: 1.7514
Epoch 4/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2608 - loss: 1.8510 - val_accuracy: 0.1300 - val_loss: 1.7364
Epoch 5/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1918 - loss: 1.9079 - val_accuracy: 0.1300 - val_loss: 1.7223
Epoch 6/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2467 - loss: 1.8425 - val_accuracy: 0.1300 - val_loss: 1.7106
Epoch 7/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2343 - loss: 1.8512 - val_accuracy: 0.4824 - val_loss: 1.7012
Epoch 8/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3061 - loss: 1.7539 - val_accuracy: 0.4720 - 

In [63]:
loss, acc = cnn_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 23.33%


## RNN

In [64]:
rnn_model = Sequential([
        Input(shape=(max_len,)),
        Embedding(VOCAB_SIZE, EMBEDDING_DIM),

        SimpleRNN(64, activation= 'tanh'),
        Dropout(0.3),
        
        Dense(32, activation='relu'),
        Dropout(0.4),
        
        Dense(6, activation='softmax')
    ])

rnn_model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=1e-4), metrics=['accuracy'])

rnn_model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ (None, 41, 28)         │       109,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 64)             │         5,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 117,878 (460.46 KB)

 Trainable params: 117,878 (460.46 KB)

 Non-trainable params: 0 (0.00 B)

In [65]:
rnn_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.8)

Epoch 1/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.1958 - loss: 1.7808 - val_accuracy: 0.3687 - val_loss: 1.7347
Epoch 2/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2562 - loss: 1.7434 - val_accuracy: 0.4187 - val_loss: 1.7035
Epoch 3/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2668 - loss: 1.7248 - val_accuracy: 0.4460 - val_loss: 1.6847
Epoch 4/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2745 - loss: 1.7500 - val_accuracy: 0.4590 - val_loss: 1.6773
Epoch 5/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3091 - loss: 1.7134 - val_accuracy: 0.4688 - val_loss: 1.6642
Epoch 6/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3196 - loss: 1.7108 - val_accuracy: 0.4694 - val_loss: 1.6510
Epoch 7/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3352 - loss: 1.6994 - val_accuracy: 0.4701 - val_loss: 1.6391
Epoch 8/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3111 - loss: 1.7135 - val_accuracy: 0.4701 - 

In [66]:
loss, acc = rnn_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 22.33%


## LSTM

In [67]:
lstm_model = Sequential([
    Input(shape=(max_len,)),
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=max_len),

    LSTM(64, activation= 'tanh'),
    Dropout(0.3),

    Dense(32, activation='relu'),
    Dropout(0.4),

    Dense(6, activation='softmax')
])

lstm_model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=1e-4), metrics=['accuracy'])

lstm_model.summary()

/opt/anaconda3/envs/yt_env/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_11 (Embedding)        │ (None, 41, 28)         │       109,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 64)             │        23,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 135,734 (530.21 KB)

 Trainable params: 135,734 (530.21 KB)

 Non-trainable params: 0 (0.00 B)

In [68]:
lstm_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.8)

Epoch 1/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.2543 - loss: 1.7893 - val_accuracy: 0.4688 - val_loss: 1.7841
Epoch 2/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2776 - loss: 1.7878 - val_accuracy: 0.4714 - val_loss: 1.7791
Epoch 3/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3259 - loss: 1.7805 - val_accuracy: 0.4720 - val_loss: 1.7720
Epoch 4/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3324 - loss: 1.7773 - val_accuracy: 0.4720 - val_loss: 1.7623
Epoch 5/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3895 - loss: 1.7641 - val_accuracy: 0.4720 - val_loss: 1.7467
Epoch 6/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3700 - loss: 1.7585 - val_accuracy: 0.4720 - val_loss: 1.7231
Epoch 7/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3714 - loss: 1.7340 - val_accuracy: 0.4720 - val_loss: 1.6797
Epoch 8/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3506 - loss: 1.7332 - val_accuracy: 0.

In [69]:
loss, acc = lstm_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 21.67%


### Attention + LSTM

In [72]:
inputs = Input(shape=(max_len,))

# 1. Embedding
x = Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=max_len)(inputs)

# 2. LSTM layer with return_sequences=True
lstm_out = LSTM(64, activation='tanh', return_sequences=True)(x)

# 3. Self-attention: query=key=value from LSTM output
attn_out = Attention(use_scale=True)([lstm_out, lstm_out])

# 4. Flatten the attended sequence into a single vector
context = GlobalAveragePooling1D()(attn_out)

# 5. Dense layers
h = Dense(32, activation='relu')(context)
h = Dropout(0.4)(h)
outputs = Dense(6, activation='softmax')(h)

at_lstm_model = Model(inputs, outputs)
at_lstm_model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
at_lstm_model.summary()

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_13      │ (None, 41)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_13        │ (None, 41, 28)    │    109,648 │ input_layer_13[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_6 (LSTM)       │ (None, 41, 64)    │     23,808 │ embedding_13[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_3         │ (None, 41, 64)    │          1 │ lstm_6[0][0],     │
│ (Attention)         │                   │            │ lstm_6[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ attention_3[0][0] │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 32)        │      2,080 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_23          │ (None, 32)        │          0 │ dense_26[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_27 (Dense)    │ (None, 6)         │        198 │ dropout_23[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 135,735 (530.21 KB)

 Trainable params: 135,735 (530.21 KB)

 Non-trainable params: 0 (0.00 B)

In [73]:
at_lstm_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split = 0.8)

Epoch 1/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.3007 - loss: 1.7885 - val_accuracy: 0.4720 - val_loss: 1.7770
Epoch 2/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3452 - loss: 1.7823 - val_accuracy: 0.4720 - val_loss: 1.7655
Epoch 3/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3271 - loss: 1.7754 - val_accuracy: 0.4720 - val_loss: 1.7521
Epoch 4/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3480 - loss: 1.7648 - val_accuracy: 0.4720 - val_loss: 1.7339
Epoch 5/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3242 - loss: 1.7599 - val_accuracy: 0.4720 - val_loss: 1.7109
Epoch 6/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3653 - loss: 1.7411 - val_accuracy: 0.4720 - val_loss: 1.6769
Epoch 7/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3366 - loss: 1.7345 - val_accuracy: 0.4720 - val_loss: 1.6420
Epoch 8/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3699 - loss: 1.6926 - val_accuracy: 0.

In [74]:
loss, acc = at_lstm_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 22.50%
